<a href="https://colab.research.google.com/github/Anannya-Vyas/EXPLORATORY-DATA-ANALYSIS/blob/main/EDA_PPT_GENERAATOR_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══════════════════════════════════════════════════════════════
# COMPLETE EDA PPT GENERATOR - ALL IN ONE CELL!
# Just run this cell and get your presentation!
# ═══════════════════════════════════════════════════════════════

# STEP 1: Install libraries (takes 30 seconds)
print("📦 Installing libraries...")
!pip install python-pptx -q
!pip install Pillow -q
print("✅ Installation complete!\n")

# STEP 2: Import everything
print("📚 Importing libraries...")
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries imported!\n")

# STEP 3: Helper Functions
def set_slide_title(slide, title_text, font_size=32):
    title = slide.shapes.title
    title.text = title_text
    title.text_frame.paragraphs[0].font.size = Pt(font_size)
    title.text_frame.paragraphs[0].font.bold = True
    title.text_frame.paragraphs[0].font.color.rgb = RGBColor(46, 134, 171)
    return title

def add_bullet_points(slide, bullet_points, left=Inches(0.5), top=Inches(1.5),
                     width=Inches(4.5), height=Inches(4)):
    textbox = slide.shapes.add_textbox(left, top, width, height)
    text_frame = textbox.text_frame
    text_frame.word_wrap = True

    for i, bullet in enumerate(bullet_points):
        if i == 0:
            p = text_frame.paragraphs[0]
        else:
            p = text_frame.add_paragraph()
        p.text = bullet
        p.level = 0
        p.font.size = Pt(18)
        p.space_after = Pt(12)
    return textbox

def save_plot_to_image(fig, filename):
    fig.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return filename

def add_image_to_slide(slide, image_path, left=Inches(5.5), top=Inches(1.5), width=Inches(4)):
    try:
        pic = slide.shapes.add_picture(image_path, left, top, width=width)
        return pic
    except Exception as e:
        print(f"Error adding image: {e}")
        return None

def add_footer(slide, text, slide_number):
    footer_box = slide.shapes.add_textbox(Inches(8.5), Inches(5), Inches(1.3), Inches(0.3))
    text_frame = footer_box.text_frame
    p = text_frame.paragraphs[0]
    p.text = f"{text} | Slide {slide_number}"
    p.font.size = Pt(10)
    p.font.color.rgb = RGBColor(128, 128, 128)
    p.alignment = PP_ALIGN.RIGHT

print("✅ Helper functions created!\n")

# STEP 4: Create Sample Data
print("📊 Creating sample data...")
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'customer_id': range(1, n_samples + 1),
    'age': np.random.randint(18, 80, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'purchased': np.random.choice([0, 1], n_samples, p=[0.6, 0.4]),
    'rating': np.random.choice([1, 2, 3, 4, 5], n_samples),
    'spending_score': np.random.randint(1, 100, n_samples)
})

# Add missing values
missing_indices = np.random.choice(df.index, size=50, replace=False)
df.loc[missing_indices, 'income'] = np.nan

print(f"✅ Sample data created! Shape: {df.shape}\n")

# STEP 5: Visualization Functions
def create_anscombe_plot():
    anscombe = sns.load_dataset("anscombe")
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    datasets = ['I', 'II', 'III', 'IV']

    for i, (ax, dataset) in enumerate(zip(axes.flat, datasets)):
        data = anscombe[anscombe['dataset'] == dataset]
        ax.scatter(data['x'], data['y'], s=50, alpha=0.6)
        ax.plot(data['x'], 3 + 0.5*data['x'], 'r-', linewidth=2)
        corr = data[['x', 'y']].corr().iloc[0, 1]
        ax.set_title(f'Dataset {dataset}\nr={corr:.3f}', fontweight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.grid(True, alpha=0.3)

    plt.suptitle("Anscombe's Quartet: Same Statistics, Different Data!",
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    return save_plot_to_image(fig, 'anscombe.png')

def create_distribution_plots(df, column):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    data = df[column].dropna()

    # Histogram with KDE
    axes[0, 0].hist(data, bins=30, edgecolor='black', alpha=0.7, density=True, color='skyblue')
    data.plot(kind='kde', ax=axes[0, 0], color='red', linewidth=2)
    axes[0, 0].axvline(data.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
    axes[0, 0].axvline(data.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
    axes[0, 0].set_title('Histogram with KDE', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Box Plot
    sns.boxplot(y=data, ax=axes[0, 1], color='lightgreen')
    axes[0, 1].set_title('Box Plot', fontweight='bold')

    # Violin Plot
    sns.violinplot(y=data, ax=axes[1, 0], color='coral')
    axes[1, 0].set_title('Violin Plot', fontweight='bold')

    # Q-Q Plot
    stats.probplot(data, dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title('Q-Q Plot', fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Distribution Analysis: {column}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return save_plot_to_image(fig, f'dist_{column}.png')

def create_correlation_heatmap(df):
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    corr_matrix = df[numerical_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True, linewidths=1)
    plt.title('Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    return save_plot_to_image(fig, 'correlation.png')

def create_categorical_analysis(df, column):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    value_counts = df[column].value_counts()

    # Bar Plot
    axes[0, 0].bar(range(len(value_counts)), value_counts.values, color='steelblue', edgecolor='black')
    axes[0, 0].set_xticks(range(len(value_counts)))
    axes[0, 0].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[0, 0].set_title('Frequency Bar Chart', fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    # Pie Chart
    axes[0, 1].pie(value_counts.values, labels=value_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0, 1].set_title('Distribution Pie Chart', fontweight='bold')

    # Horizontal Bar
    axes[1, 0].barh(range(len(value_counts)), value_counts.values, color='coral', edgecolor='black')
    axes[1, 0].set_yticks(range(len(value_counts)))
    axes[1, 0].set_yticklabels(value_counts.index)
    axes[1, 0].set_title('Horizontal Bar Chart', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='x')

    # Stats table
    axes[1, 1].axis('off')
    freq_table = pd.DataFrame({
        'Category': value_counts.index,
        'Count': value_counts.values,
        'Percentage': [f'{v/len(df)*100:.1f}%' for v in value_counts.values]
    })
    table = axes[1, 1].table(cellText=freq_table.values, colLabels=freq_table.columns,
                            cellLoc='center', loc='center', colWidths=[0.3, 0.3, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    plt.suptitle(f'Categorical Analysis: {column}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return save_plot_to_image(fig, f'cat_{column}.png')

print("✅ Visualization functions created!\n")

# STEP 6: CREATE PRESENTATION
print("🎨 Creating PowerPoint Presentation...")
print("="*60)

prs = Presentation()
prs.slide_width = Inches(10)
prs.slide_height = Inches(5.625)

slide_num = 0

# SLIDE 1: Title
slide_num += 1
print(f"Creating Slide {slide_num}: Title")
slide = prs.slides.add_slide(prs.slide_layouts[0])
title = slide.shapes.title
subtitle = slide.placeholders[1]
title.text = "EXPLORATORY DATA ANALYSIS"
title.text_frame.paragraphs[0].font.size = Pt(44)
title.text_frame.paragraphs[0].font.bold = True
title.text_frame.paragraphs[0].font.color.rgb = RGBColor(46, 134, 171)
subtitle.text = "From Fundamentals to Advanced Techniques\n\nYour Name\nAI/ML Training Course\n2024"

# SLIDE 2: Table of Contents
slide_num += 1
print(f"Creating Slide {slide_num}: Table of Contents")
slide = prs.slides.add_slide(prs.slide_layouts[1])
set_slide_title(slide, "Table of Contents")
toc = [
    "1. Introduction to EDA",
    "2. EDA Philosophy & Anscombe's Quartet",
    "3. Data Types & First Look",
    "4. Distribution Analysis",
    "5. Correlation Analysis",
    "6. Categorical Analysis",
    "7. Best Practices & Pitfalls",
    "8. Tools & Resources",
    "9. Key Takeaways"
]
add_bullet_points(slide, toc, left=Inches(1), top=Inches(1.5), width=Inches(8.5), height=Inches(4))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 3: What is EDA
slide_num += 1
print(f"Creating Slide {slide_num}: What is EDA")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "What is Exploratory Data Analysis?")
points = [
    "📊 Definition:",
    "Critical process of initial data investigation",
    "",
    "🎯 Key Objectives:",
    "🔍 Understand data structure",
    "📈 Identify patterns",
    "🎯 Detect outliers",
    "💡 Generate hypotheses",
    "🧹 Guide preprocessing"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(9), height=Inches(4))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 4: Anscombe's Quartet
slide_num += 1
print(f"Creating Slide {slide_num}: Anscombe's Quartet")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Anscombe's Quartet: Always Visualize!")
points = [
    "🔥 Hidden Gem:",
    "",
    "All 4 datasets have:",
    "• Same mean",
    "• Same variance",
    "• Same correlation",
    "",
    "But COMPLETELY",
    "different distributions!",
    "",
    "💡 Never trust",
    "statistics alone!"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(3.5), height=Inches(4))
img = create_anscombe_plot()
add_image_to_slide(slide, img, left=Inches(4.2), top=Inches(1.3), width=Inches(5.5))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 5: Data Types
slide_num += 1
print(f"Creating Slide {slide_num}: Data Types")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Understanding Data Types")
textbox = slide.shapes.add_textbox(Inches(0.5), Inches(1.5), Inches(9), Inches(4))
text_frame = textbox.text_frame
text_frame.text = """
📊 CATEGORICAL (Qualitative):
   ├─ Nominal: Colors, Gender, Country
   ├─ Ordinal: Education Level, Ratings
   └─ Binary: Yes/No, True/False

📈 NUMERICAL (Quantitative):
   ├─ Discrete: Count data (0,1,2,3...)
   └─ Continuous: Measurements
       ├─ Interval: Temperature °C
       └─ Ratio: Height, Weight

🔥 Don't treat ordinal as nominal - BIG MISTAKE!
"""
text_frame.paragraphs[0].font.size = Pt(16)
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 6: First Look
slide_num += 1
print(f"Creating Slide {slide_num}: First Look")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "First Look at Data")
points = [
    "The 5-Step Ritual:",
    "",
    "1️⃣ Check shape & size",
    "2️⃣ Examine dtypes",
    "3️⃣ Preview rows",
    "4️⃣ Calculate statistics",
    "5️⃣ Find missing values",
    "",
    f"Dataset: {df.shape}",
    f"Missing: {df.isnull().sum().sum()}"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(4), height=Inches(4))
code_box = slide.shapes.add_textbox(Inches(5), Inches(1.5), Inches(4.5), Inches(4))
code_box.text_frame.text = "df.shape\ndf.info()\ndf.head()\ndf.describe()\ndf.isnull().sum()"
code_box.text_frame.paragraphs[0].font.name = 'Courier New'
code_box.text_frame.paragraphs[0].font.size = Pt(14)
code_box.fill.solid()
code_box.fill.fore_color.rgb = RGBColor(240, 240, 240)
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 7: Distribution Analysis
slide_num += 1
print(f"Creating Slide {slide_num}: Distribution Analysis")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Distribution Analysis")
points = [
    "📊 Key Metrics:",
    "",
    f"Mean: {df['age'].mean():.2f}",
    f"Median: {df['age'].median():.2f}",
    f"Std: {df['age'].std():.2f}",
    f"Skew: {df['age'].skew():.2f}",
    "",
    "Check for:",
    "• Normality",
    "• Outliers",
    "• Symmetry"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(3.5), height=Inches(4))
img = create_distribution_plots(df, 'age')
add_image_to_slide(slide, img, left=Inches(4.2), top=Inches(1.3), width=Inches(5.5))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 8: Correlation
slide_num += 1
print(f"Creating Slide {slide_num}: Correlation Analysis")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Correlation Analysis")
points = [
    "🔗 Types:",
    "",
    "• Pearson",
    "  (Linear)",
    "",
    "• Spearman",
    "  (Monotonic)",
    "",
    "• Kendall",
    "  (Robust)",
    "",
    "Use all three!"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(3.5), height=Inches(4))
img = create_correlation_heatmap(df)
add_image_to_slide(slide, img, left=Inches(4.2), top=Inches(1.3), width=Inches(5.5))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 9: Categorical Analysis
slide_num += 1
print(f"Creating Slide {slide_num}: Categorical Analysis")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Categorical Analysis")
points = [
    "📊 Steps:",
    "",
    "1️⃣ Frequencies",
    "2️⃣ Percentages",
    "3️⃣ Mode",
    "4️⃣ Entropy",
    "",
    f"Categories: {df['education'].nunique()}",
    f"Mode: {df['education'].mode()[0]}"
]
add_bullet_points(slide, points, left=Inches(0.5), top=Inches(1.5), width=Inches(3.5), height=Inches(4))
img = create_categorical_analysis(df, 'education')
add_image_to_slide(slide, img, left=Inches(4.2), top=Inches(1.3), width=Inches(5.5))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 10: Best Practices
slide_num += 1
print(f"Creating Slide {slide_num}: Best Practices")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "EDA Best Practices")
points = [
    "✅ Always visualize first",
    "✅ Use multiple methods",
    "✅ Check assumptions",
    "✅ Document findings",
    "✅ Question anomalies",
    "",
    "❌ Don't auto-remove outliers",
    "❌ Don't trust stats alone",
    "❌ Don't ignore missing patterns",
    "❌ Don't skip visualization"
]
add_bullet_points(slide, points, left=Inches(1), top=Inches(1.5), width=Inches(8), height=Inches(4))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 11: Tools
slide_num += 1
print(f"Creating Slide {slide_num}: Tools")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Automated EDA Tools")
points = [
    "🔥 Top Tools:",
    "",
    "1️⃣ Pandas Profiling",
    "2️⃣ Sweetviz",
    "3️⃣ AutoViz",
    "4️⃣ D-Tale",
    "5️⃣ DataPrep",
    "",
    "All generate HTML reports",
    "with one line of code!"
]
add_bullet_points(slide, points, left=Inches(1), top=Inches(1.5), width=Inches(8), height=Inches(4))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 12: Key Takeaways
slide_num += 1
print(f"Creating Slide {slide_num}: Key Takeaways")
slide = prs.slides.add_slide(prs.slide_layouts[5])
set_slide_title(slide, "Key Takeaways")
points = [
    "🎯 THE GOLDEN RULES:",
    "",
    "1️⃣ 👁️ ALWAYS VISUALIZE",
    "",
    "2️⃣ 🔍 QUESTION EVERYTHING",
    "",
    "3️⃣ 📊 USE MULTIPLE METHODS",
    "",
    "4️⃣ 📝 DOCUMENT FINDINGS",
    "",
    "5️⃣ 🔄 EDA IS ITERATIVE"
]
add_bullet_points(slide, points, left=Inches(1), top=Inches(1.5), width=Inches(8), height=Inches(4))
add_footer(slide, "EDA Presentation", slide_num)

# SLIDE 13: Thank You
slide_num += 1
print(f"Creating Slide {slide_num}: Thank You")
slide = prs.slides.add_slide(prs.slide_layouts[0])
title = slide.shapes.title
subtitle = slide.placeholders[1]
title.text = "THANK YOU!"
title.text_frame.paragraphs[0].font.size = Pt(54)
title.text_frame.paragraphs[0].font.bold = True
title.text_frame.paragraphs[0].font.color.rgb = RGBColor(46, 134, 171)
subtitle.text = "Questions?\n\nyour.email@example.com"

# SAVE
filename = "EDA_Presentation.pptx"
prs.save(filename)

print("="*60)
print(f"✅ SUCCESS! Presentation created!")
print(f"📄 File: {filename}")
print(f"📊 Total Slides: {slide_num}")
print("="*60)

# Download (for Colab)
try:
    from google.colab import files
    print("\n📥 Downloading...")
    files.download(filename)
    print("✅ Check your Downloads folder!")
except:
    print(f"\n💾 File saved: {filename}")

print("\n🎉 ALL DONE! Present with confidence! 💪")

📦 Installing libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.8 MB/s eta 0:00:00
✅ Installation complete!

📚 Importing libraries...
✅ Libraries imported!

✅ Helper functions created!

📊 Creating sample data...
✅ Sample data created! Shape: (1000, 8)

✅ Visualization functions created!

🎨 Creating PowerPoint Presentation...
Creating Slide 1: Title
Creating Slide 2: Table of Contents
Creating Slide 3: What is EDA
Creating Slide 4: Anscombe's Quartet
Creating Slide 5: Data Types
Creating Slide 6: First Look
Creating Slide 7: Distribution Analysis
Creating Slide 8: Correlation Analysis
Creating Slide 9: Categorical Analysis
Creating Slide 10: Best Practices
Creating Slide 11: Tools
Creating Slide 12: Key Takeaways
Creating Slide 13: Thank You
✅ SUCCESS! Presentation created!
📄 File: EDA_Presentation.pptx
📊 Total Slides: 13

📥 Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Check your Downloads folder!

🎉 ALL DONE! Present with confidence! 💪
